# Comparing parallel universes - ecoHeat
When a user has ecoHeat OFF how much could they be saving?
e.g. This is what ecoHeat ON could save you (in a week? day? year?)

In [ ]:
# Imports
import ast
import pandas as pd
import requests
from datetime import datetime, timezone, timedelta

from pathlib import Path
from scipy.integrate import trapezoid

from model_helper_functions import get, plot_temp_power

In [ ]:
# Set up API call
url = "https://bristol.passivuk.com/optimisation"
with open("API_key.txt", "r") as file:
    API_key = file.read()

#### Research question:
If the user had had ecoHeat enabled - could they have saved money?

## Read in and store data
Must filter data to houses with viable contents for problem we are asking

Data must have: 
- input power
- room temperature
- external temperature
- tariff
- ecoheat indicator (0/1)

Also must:
- Sometimes have ecoHeat disabled (indicator = 0)

In [ ]:
# load event files
event_path = Path("data/PSTData5_reformat_v3")
event_files = event_path.glob("*.csv")

houses_events = {}
for file in event_files:
    house_id = file.stem.split("_")[0]
    houses_events[house_id] = pd.read_csv(file)
    houses_events[house_id]["Timestamp"] = pd.to_datetime(houses_events[house_id]["Timestamp"])

In [ ]:
# Read in, sort and store data to use for problem
temp_path = Path("data/PSTData5")
temp_files = temp_path.glob("*.csv")

houses_temps = {}
houses_without_power = []
houses_without_setpoints = []
houses_without_room_temp = []
houses_without_ext_temp = []
houses_without_tariff = []
houses_without_ecoheat_indicator = []
good_houses_ecoheat_always = []
zone_2 = True
for file in temp_files:
    if "Events" not in str(file):
        house_id = file.stem.split("_")[0]
        data = pd.read_csv(file)
        temp = {}
        temp["Datetimes"] = pd.to_datetime(data["Time (UTC)"])
        temp["Setpoint Z1"] = data["Setpoint temperature (Zone 1) (°C)"]
        temp["Room Temp Z1"] = data["Room temperature (Zone 1) (°C)"]
        try:
            temp["Setpoint Z2"] = data["Setpoint temperature (Zone 2) (°C)"]
            temp["Room Temp Z2"] = data["Room temperature (Zone 2) (°C)"]
        except KeyError:
            print("Data doesn't have zone 2")
            zone_2 = False
        temp["External Temp"] = data["External temperature (°C)"]
        temp["Input Power"] = data["Power consumption (kW)"]
        temp["Elec Cost (p/kWh)"] = data["Tariff rate (p/kWh)"]
        temp["Ecoheat (0/1)"] = data["Ecoheat (0/1)"]

        # Remove NaNs
        house_df = pd.DataFrame(temp).dropna(axis=1, how="all")  # Drop any columns where all values are NaN
        house_df = house_df.dropna(axis=0, how="any")  # Drop any rows where one or more values are NaN
        
        # Store data
        if "Input Power" not in house_df.columns:
            houses_without_power.append(house_id)
        elif "Setpoint Z1" not in house_df.columns:
            houses_without_setpoints.append(house_id)
        elif "Room Temp Z1" not in house_df.columns:
            houses_without_room_temp.append(house_id)
        elif "External Temp" not in house_df.columns:
            houses_without_ext_temp.append(house_id)
        elif "Elec Cost (p/kWh)" not in house_df.columns:
            houses_without_tariff.append(house_id)
        elif "Ecoheat (0/1)" not in house_df.columns:
            houses_without_ecoheat_indicator.append(house_id)
        elif sum(house_df['Ecoheat (0/1)'] == 0) == 0:
            good_houses_ecoheat_always.append(house_id)
        else:
            houses_temps[house_id] = house_df

print("No power: ", houses_without_power)
print("No setpoints: ", houses_without_setpoints)
print("No room temp: ", houses_without_room_temp)
print("No ext temp: ", houses_without_ext_temp)
print("No tariff: ", houses_without_tariff)
print("No Ecoheat: ", houses_without_ecoheat_indicator)
print("Viable data - ecoHeat never disabled: ", good_houses_ecoheat_always)
print("Data to use: ", list(houses_temps.keys()))

In [ ]:
event_df = houses_events["30055"]
temp_df = houses_temps["30055"]

In [ ]:
event_df

In [ ]:
temp_df

### Understand house

In [ ]:
print("Number of datapoints: ", len(temp_df))
print("Number of datapoints with Ecoheat disabled:", sum(temp_df['Ecoheat (0/1)'] == 0))

In [ ]:
start_index = 1871
end_index = int(start_index+(30*60/5))
# NOTE: This index corresponds to the 31/12/2025, which was determined not to have any overrides
# made to the schedule for 30 hours using the "eda_tempoverride" notebook. 
# This simplifies the situation and allows a closer comparison between the original data and the model API output
temp_df.loc[start_index:end_index]

In [ ]:
if zone_2:
    num_zones = 2

plot_temp_power(temp_df.loc[start_index:end_index], num_zones, 2, share_x=False, plot_tariff=False)

### Extract schedule

In [ ]:
# Filter to schedules
mask = event_df["Type"] == "Schedule"
schedules = event_df.loc[mask, ["Timestamp", "Payload"]]

# Preprocess data: remove non-heating schedules and format payloads
schedules_dict_z1 = {}
if zone_2:
    schedules_dict_z2 = {}
for _, row in schedules.iterrows():
    time = row.get("Timestamp")
    data = row.get("Payload")
    if isinstance(data, str): # if payload is string, convert to dict
        data = ast.literal_eval(data)
    if data.get("type") == "heating" and data.get("zone") == 1: # only use heating schedules
        schedules_dict_z1[time] = data.get("schedule")
    elif zone_2 and data.get("type") == "heating" and data.get("zone") == 2:
        schedules_dict_z2[time] = data.get("schedule")

# Time at start of data range
time = temp_df.loc[start_index]["Datetimes"]
print("Start time in temperature data: ", time)
days_of_week = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]
weekday = days_of_week[time.weekday()]
print("Start time corresponding weekday: ",weekday)

# Extract time nearest (before) or equal to time from dataset to find applicable schedule
nearest_time = min((t for t in schedules_dict_z1.keys() if t <= time), key=lambda t: time - t)
print("Last time that a zone 1 schedule was set: ", nearest_time)
applicable_schedule_z1 = schedules_dict_z1[nearest_time][weekday]
print("Corresponding zone 1 schedule: ", applicable_schedule_z1)
if zone_2:
    nearest_time = min((t for t in schedules_dict_z2.keys() if t <= time), key=lambda t: time - t)
    print("Last time that a zone 2 schedule was set: ", nearest_time)
    applicable_schedule_z2 = schedules_dict_z2[nearest_time][weekday]
    print("Corresponding zone 2 schedule: ", applicable_schedule_z2)

# NOTE: applicable schedule not quite right for 30 hour generations will be that day and the next, but model API too simple and can't have distinct days

Problem: Overrides in data mean that the true schedule doesn't always align with the listed schedules.

This also means that heatpump is tasks with new information immediately.. this is a different question (see Priya's work)

Simplifying assumption: run for just one part of the data, where no overides occurred!

I have used the "eda_tempoverride" notebook in the EDA folder to locate a datetime (31st Dec 2025) where for 30 hours there was no override in the data fro house 30055, and for this proof of concept I will use that time. This could be automated in the future.

# Call API

Need to match:
- start time
- number of zones in house
- start room temperature (z1 and z2)
- schedules (z1 and z2)

Don't need to match:
- External temperature ? - Thinking is that the model only ever uses predicted temperature not real time sensor measurements? Need to check this with Rosie and Edwin for accuracy

Other factors not considered in current pipeline:
- Might want to parallel hot water behaviour also. Default is an empty schedule in model API, but a hot water schedule has been set for the hosue we are using (30055).
    - Blocker: would need to reformat hot water schedules in data, currently only done for heating.

In [ ]:
# Reformat time for model API
formatted_time = time.strftime("%Y-%m-%dT%H:%M:%SZ")
formatted_time

In [ ]:
# Set up API call
if zone_2:
    api_input = {
    "num_zones": 2,
    "zone_1_heating_daily_schedule": applicable_schedule_z1,
    "zone_2_heating_daily_schedule": applicable_schedule_z2,
    "start_datetime": formatted_time,
    "zone_1_temperature": temp_df.loc[start_index]["Room Temp Z1"],
    "zone_2_temperature": temp_df.loc[start_index]["Room Temp Z2"],
    }
else:
    api_input = {
        "num_zones": 1,
        "zone_1_heating_daily_schedule": applicable_schedule_z1,
        "start_datetime": formatted_time,
        "zone_1_temperature": temp_df.loc[start_index]["Room Temp Z1"],
    }

In [ ]:
return_1 = requests.post(url, headers={"X-API-Key": API_key}, json=api_input).json()

### Plot results

In [ ]:
if return_1['success']:
    dts = [datetime.fromisoformat(t.replace("Z", "+00:00")) for t in return_1["real_dt"]]
    n = len(dts)
    S = return_1["State"]
    if zone_2:
        output_data = pd.DataFrame({
            "Datetimes": dts,
            "Setpoint Z1": get(S, n, "setpoint.z1"),
            "Setpoint Z2": get(S, n, "setpoint.z2"),
            "Room Temp Z1": get(S, n, "room_temp.z1"),
            "Room Temp Z2": get(S, n, "room_temp.z2"),
            "External Temp": get(S, n, "ext"),
            "Input Power": get(S, n, "E.hs1.heat.z1"),
            "Output Power": get(S, n, "U.hs1.z1"),
            "Elec Cost (p/kWh)": get(S, n, "elec_cost"),
            "Tank Temp": get(S, n, "tank_temp"),
            "HW Setpoint": get(S, n, "setpoint_hw")
            # "Tariff":get(S, n, "tariff")
        })
    else:
        output_data = pd.DataFrame({
            "Datetimes": dts,
            "Setpoint Z1": get(S, n, "setpoint.z1"),
            "Room Temp Z1": get(S, n, "room_temp.z1"),
            "External Temp": get(S, n, "ext"),
            "Input Power": get(S, n, "E.hs1.heat.z1"),
            "Output Power": get(S, n, "U.hs1.z1"),
            "Elec Cost (p/kWh)": get(S, n, "elec_cost"),
            "Tank Temp": get(S, n, "tank_temp"),
            "HW Setpoint": get(S, n, "setpoint_hw")
            # "Tariff":get(S, n, "tariff")
        })
    plot_temp_power(output_data, api_input.get("num_zones"), 2, share_x=False, plot_tariff=False,
                    z1_setpoints=temp_df["Setpoint Z1"].loc[start_index:end_index],
                    z2_setpoints=temp_df["Setpoint Z2"].loc[start_index:end_index],
                    setpoint_dts=temp_df["Datetimes"].loc[start_index:end_index])

### Calculate energy costs

In [ ]:
def get_time_passed(df, start_time=datetime(year=2024, month=1, day=1, hour=0)):
    """Convert time array (in real time) to time passed in hours since a set start time
    
    Args:
        df (Dataframe): data containing time under Datetimes column
        start_time (datetime): starting datetime to calculate from.
            Defaults to midnight on 01/01/2024

    Returns:
        time_in_hours (array): array of time passed in hours since a set start time,
            whose intervals align with the original Datetimes values in df.
    """
    time_in_hours = []
    try:
        df.Datetimes.iloc[0]-start_time
    except TypeError: # timezone awareness in model API
        # Error message: Cannot subtract tz-naive and tz-aware datetime-like objects
        # Have to convert start time to tz-aware time if needed
        start_time = start_time.replace(tzinfo=timezone(timedelta(hours=0, minutes=0)))

    for time in df.Datetimes:
        time_elapsed = time-start_time  # Time deltas store time in days and seconds (for reasons....)
        time_in_hours.append(time_elapsed.days*24 + time_elapsed.seconds/(60*60))  # Time in hours since the start!
    return time_in_hours

In [ ]:
def calc_power_energy_cost(df, start_time=datetime(year=2024, month=1, day=1, hour=0)):
    """Calculate total power used, energy used and cost of a 30 hour period
    
    Args:
        df (Dataframe): data containing time under Datetimes column
        start_time (datetime): starting datetime to calculate from.
            Defaults to midnight on 01/01/2024

    Returns:
        energy_used (float): energy used across 30 hours (kWh)
        cost (float): cost in pounds of the energy used in 30 hours
    """
    power_df = df.copy(deep=True).dropna()
    time_in_hours = get_time_passed(power_df, start_time)
    if len(set(power_df["Elec Cost (p/kWh)"])) == 1:
        energy_used = trapezoid(power_df["Input Power"], x=time_in_hours)
        print("Total energy used: ", round(energy_used, 2), " kWh")
        cost = round(energy_used*power_df["Elec Cost (p/kWh)"].iloc[1]/100, 2)
        print("Energy cost: £", cost)
    return energy_used, cost

In [ ]:
# Model API calculation
print("With ecoHeat on (model API):")
E_API, C_API = calc_power_energy_cost(output_data)

In [ ]:
# Original data calculation:
print("With ecoHeat off (real data):")
E_orig, C_orig = calc_power_energy_cost(temp_df.loc[start_index:start_index+(30*60/5)])

In [ ]:
print("Savings with ecoHeat turned on:")
print("Per 30 hour simulation: £", round((C_orig - C_API), 2))
print("Per day (24 hours): £", round((C_orig - C_API)*24/30, 2))
print("Per week (7 days): £", round((C_orig - C_API)*24/30*7, 2))
print("Per year (365 days): £", round((C_orig - C_API)*24/30*365, 2))